# Debugging U-Net

Open submission.py in a second JupyterLab tab. Edit that file, save it, and rerun these checks. The provided functions intentionally need your repair. Read Statement.md for the source correction and exact requirements.

In [ ]:
import importlib
import torch
import submission
torch.set_num_threads(2)
importlib.reload(submission)
model = submission.BrokenUNet()
try:
    with torch.no_grad():
        output = model(torch.randn(1, 3, 256, 256))
    print('Output shape:', tuple(output.shape))
    assert tuple(output.shape) == (1, 2, 110, 110)
except Exception as error:
    print(type(error).__name__ + ':', error)

## Check center cropping

The values matter as well as the shape. A resize is not a crop.

In [ ]:
importlib.reload(submission)
x = torch.arange(63).reshape(1, 1, 7, 9)
try:
    cropped = submission.BrokenUNet().center_crop(x, (1, 1, 3, 5))
    assert torch.equal(cropped, x[:, :, 2:5, 2:7])
    print('Center crop passed')
except Exception as error:
    print(type(error).__name__ + ':', error)

## Check the metric

The second case below corrects an error in the original source self-check. A predicted class with no true pixels still has a nonzero union.

In [ ]:
importlib.reload(submission)
cases = [(torch.tensor([[[0, 0], [1, 1]]]), torch.tensor([[[0, 1], [1, 1]]]), 7 / 12),
         (torch.tensor([[[0, 0], [0, 1]]]), torch.zeros((1, 2, 2), dtype=torch.long), 0.375),
         (torch.zeros((1, 2, 2), dtype=torch.long), torch.zeros((1, 2, 2), dtype=torch.long), 1.0)]
for predicted, truth, expected in cases:
    try:
        actual = float(submission.calculate_miou(predicted, truth, 2))
        print('Got', actual, 'expected', expected)
        assert abs(actual - expected) < 1e-6
    except Exception as error:
        print(type(error).__name__ + ':', error)

## Submit and explain

Save submission.py, then upload it on the hub task page. Keep your reasoning here: why was the skip larger, what does cropping discard, and why is accuracy insufficient for this task?